# LangGraph Agent Demo

Multi-agent pipeline: **orchestrator → researcher → human review → writer**

Two versions available:
- `langgraph_agent.py` — OpenAI `gpt-4o-mini`
- `langgraph_agent_groq.py` — Groq `llama-3.3-70b-versatile` via Cloudflare Gateway

**Why not call `main()`?**  
`main()` uses `argparse` which reads `sys.argv`. In Jupyter, `sys.argv` contains notebook kernel arguments and causes argparse to error. Call `run_interactive()` or `run_noninteractive()` directly instead.

In [ ]:
import sys
sys.path.insert(0, r"C:\python")  # ensure project root is on path

## Pick your LLM backend
Uncomment one import block.

In [ ]:
# ── Option A: OpenAI gpt-4o-mini ─────────────────────────────────────────────
from projects.langgraph_agent.langgraph_agent import run_interactive, run_noninteractive

# ── Option B: Groq llama-3.3-70b via Cloudflare Gateway ──────────────────────
# from projects.langgraph_agent.langgraph_agent_groq import run_interactive, run_noninteractive

## Option 1 — Interactive (human review loop)

Pauses at the review step and waits for your input in the cell output.  
Type `ok` to approve, or type revision instructions to send the researcher back.

In [ ]:
TASK = "What are the top 3 programming languages to learn in 2025 and why?"

draft = run_interactive(TASK, thread_id="notebook-1")

## Option 2 — Non-interactive (pre-supply feedback)

Useful for automated runs or when you want to skip the review step.  
Pass `feedback="ok"` to auto-approve, or pass revision instructions.

In [ ]:
TASK     = "Explain the main differences between REST and GraphQL APIs"
FEEDBACK = "ok"   # auto-approve — or: "focus more on performance trade-offs"

draft = run_noninteractive(TASK, feedback=FEEDBACK, thread_id="notebook-2")
print(draft)

## Option 3 — Render the final draft as Markdown

In [ ]:
from IPython.display import Markdown, display

TASK = "What is the difference between supervised and unsupervised machine learning?"

draft = run_noninteractive(TASK, feedback="ok", thread_id="notebook-3")
display(Markdown(draft))

## Option 4 — Inspect the full graph state after a run

In [ ]:
from projects.langgraph_agent.langgraph_agent_groq import build_graph
from langgraph.types import Command

TASK = "What is LangGraph and when should I use it?"

graph  = build_graph()
config = {"configurable": {"thread_id": "notebook-inspect"}}

initial_state = {
    "task": TASK, "framed_task": "", "research": "",
    "draft": "", "feedback": "", "revision": 0,
}

# Run to the interrupt
for _ in graph.stream(initial_state, config, stream_mode="values"):
    pass

# Resume with approval
for _ in graph.stream(Command(resume="ok"), config, stream_mode="values"):
    pass

# Inspect every field in the final state
state = graph.get_state(config).values
for key, value in state.items():
    print(f"\n{'='*40}")
    print(f"[{key.upper()}]")
    print(f"{'='*40}")
    print(value if value else "(empty)")